# Wrapper Architecture

In [7]:
import numpy as np
from torch.nn.utils import clip_grad_norm_
import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import accuracy_score

class MyWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, model, classes:list | np.ndarray, lr:float=1e-3, epochs:int=50, batch_size:int=None, device:torch.device=None, is_fitted:bool = False):
        if device is None:
            self.DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.DEVICE = device

        self.model = model.to(self.DEVICE)
        self.classes = classes

        self.hyperparameters = {
            "lr": lr,
            "epochs": epochs,
            "batch_size": batch_size if batch_size is not None else 32,
        }


    def fit(self, X: np.ndarray, y: np.ndarray, sample_weight=None) -> float:
        def _train_one_epoch(model, loader, optimizer, criterion, max_norm:float=1.0):
            model.train()
            running_loss = 0.0
            for X, y in loader:
                X, y = X.to(self.DEVICE), y.to(self.DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(X), y)
                running_loss += loss.item()
                loss.backward()
                clip_grad_norm_(model.parameters(), max_norm=max_norm)
                optimizer.step()
            avg_loss = running_loss / len(loader)

            return avg_loss

        self.model.to(self.DEVICE)
        optimzer = torch.optim.Adam(self.model.parameters(), lr=self.hyperparameters["lr"])

        if len(self.classes) == 2:
            criterion = nn.BCEWithLogitsLoss()
        else:
            criterion = nn.CrossEntropyLoss()

        X_tensor = torch.Tensor(X).to(self.DEVICE)
        y_tensor = torch.Tensor(y).to(self.DEVICE)

        if sample_weight is not None:
            sampler = WeightedRandomSampler(sample_weight, len(X_tensor), replacement=True)
        else:
            sampler = None

        train_loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=self.hyperparameters["batch_size"], sampler=sampler)

        running_loss = 0.0
        for epoch in range(self.hyperparameters["epochs"]):
            running_loss += _train_one_epoch(self.model, train_loader, optimzer, criterion)

        return running_loss/self.hyperparameters["epochs"]


    def predic_proba(self, X: np.ndarray) -> np.ndarray:
        output = self.model.predict(X)
        proba = torch.softmax(output, dim=1).detach().cpu().numpy()

        return proba

    def predict(self, X: np.ndarray) -> np.ndarray:
        proba = self.predic_proba(X)
        preds = proba.argmax(axis=1)

        return preds


    def score(self, X, y, sample_weight=None):
        check_is_fitted(self)
        y_pred = self.predict(X)

        return accuracy_score(y, y_pred, sample_weight=sample_weight)


# Using Wrapper

## Data

In [5]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import torch

# Data
X, y = make_classification(n_samples=5000, n_features=20, random_state=42)
X_train_val_cal, X_test, y_train_val_cal, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_val, X_cal, y_train_val, y_cal = train_test_split(X_train_val_cal, y_train_val_cal, test_size=0.2, random_state=42)

## Model

### Architecture

In [4]:
from torch import nn
from typing import Literal

class MLPClassifier(nn.Module):
    def __init__(self, n_hidden:int, hidden_dim:int, input_dim:int, output_dim:int, activation:Literal['ReLU', 'Tanh'] = 'ReLU'):
        super(MLPClassifier, self).__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(hidden_dim, output_dim)

        if activation == 'ReLU':
            self.activation = nn.ReLU()
        elif activation == 'Tanh':
            self.activation = nn.Tanh()
        else:
            raise ValueError(f"Unknown activation function: {activation}")

    def forward(self, input):
        x = self.activation(self.input_layer(input))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.output_layer(x)

        return x.squeeze(1) if x.shape[-1] == 1 else x